# Embedding & ChromaDB Indexing — RAG Soạn thảo Văn bản Hành chính

Notebook này thực hiện:
1. Embed 3 tập chunks bằng `multilingual-e5-large-instruct`
2. Nạp vào ChromaDB persistent (3 collection riêng biệt)
3. Kiểm tra retrieval thử

## Kiến trúc collections

| Collection | Source | Dùng cho |
|---|---|---|
| `legal_chunks` | `legal_chunks.parquet` | Truy xuất điều khoản pháp luật |
| `forms_chunks` | `forms_chunks.parquet` | Truy xuất template biểu mẫu |
| `examples_chunks` | `examples_chunks.parquet` | Few-shot retrieval ví dụ điền sẵn |

## Lưu ý ChromaDB & chunk_id
ChromaDB yêu cầu `id` là string không chứa ký tự đặc biệt gây lỗi parsing.  
→ `chunk_id` gốc (chứa `/`) sẽ được **sanitize** khi dùng làm Chroma ID, nhưng **giữ nguyên** trong metadata để truy xuất ngược.

---
## 0. Cài đặt thư viện

In [ ]:
# Chạy 1 lần khi setup môi trường
# !pip install chromadb sentence-transformers torch pandas pyarrow tqdm

---
## 1. Import & Cấu hình

In [ ]:
import pandas as pd
import numpy as np
import re
import json
import time
from pathlib import Path
from typing import List, Dict, Optional
from tqdm.notebook import tqdm

import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

# ── Đường dẫn ────────────────────────────────────────────────────────────────
CHUNK_DIR  = Path("dataset/chunks")
CHROMA_DIR = Path("dataset/chromadb")   # persistent storage
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

CACHE_MODEL_PATH = Path("models/embedding")

CHUNK_PATHS = {
    "legal"   : CHUNK_DIR / "legal_chunks.parquet",
    "forms"   : CHUNK_DIR / "forms_chunks.parquet",
    "examples": CHUNK_DIR / "examples_chunks.parquet",
}

# ── Tham số embedding ────────────────────────────────────────────────────────
EMBED_MODEL_NAME = "intfloat/multilingual-e5-large-instruct"
EMBED_BATCH_SIZE = 64      # giảm xuống 32 nếu OOM
EMBED_DEVICE     = "cuda" if __import__('torch').cuda.is_available() else "cpu"

# ── Tên collections trong ChromaDB ──────────────────────────────────────────
COLLECTION_NAMES = {
    "legal"   : "legal_chunks",
    "forms"   : "forms_chunks",
    "examples": "examples_chunks",
}

print(f"✅ Import thành công.")
print(f"   Device   : {EMBED_DEVICE}")
print(f"   ChromaDB : {CHROMA_DIR}")
for name, path in CHUNK_PATHS.items():
    status = "✅" if path.exists() else "❌ KHÔNG TÌM THẤY"
    print(f"   [{status}] {name}: {path}")

✅ Import thành công.
   Device   : cpu
   ChromaDB : dataset\chromadb
   [✅] legal: dataset\chunks\legal_chunks.parquet
   [✅] forms: dataset\chunks\forms_chunks.parquet
   [✅] examples: dataset\chunks\examples_chunks.parquet


---
## 2. Load Embedding Model

In [ ]:
print(f"Đang load model: {EMBED_MODEL_NAME} ...")
t0 = time.time()

embed_model = SentenceTransformer(
    EMBED_MODEL_NAME,
    device=EMBED_DEVICE,
    cache_folder= CACHE_MODEL_PATH,
)
EMBED_DIM = embed_model.get_sentence_embedding_dimension()

print(f"✅ Model loaded  ({time.time()-t0:.1f}s)")
print(f"   Embedding dim : {EMBED_DIM}")
print(f"   Max seq length: {embed_model.max_seq_length}")

Đang load model: intfloat/multilingual-e5-large-instruct ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\VITUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\VITUS\.cache\huggingface\hub\models--intfloat--multilingual-e5-large-instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_xlm-roberta_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

In [ ]:
# ── Hàm embed với instruction prefix (chuẩn E5-instruct) ─────────────────────
#
# E5-instruct yêu cầu:
#   - Khi index (document): prefix "passage: "
#   - Khi query           : prefix "Instruct: ...\nQuery: "
#
# Tham khảo: https://huggingface.co/intfloat/multilingual-e5-large-instruct

QUERY_INSTRUCTION = (
    "Instruct: Cho một câu hỏi về pháp luật Việt Nam, "
    "hãy tìm đoạn văn bản pháp luật liên quan nhất.\nQuery: "
)

def embed_passages(texts: List[str], batch_size: int = EMBED_BATCH_SIZE) -> np.ndarray:
    """Embed list văn bản để indexing (dùng prefix 'passage: ')."""
    prefixed = [f"passage: {t}" for t in texts]
    return embed_model.encode(
        prefixed,
        batch_size=batch_size,
        show_progress_bar=False,
        normalize_embeddings=True,   # cosine similarity → dot product
        convert_to_numpy=True,
    )

def embed_query(query: str) -> np.ndarray:
    """Embed câu query khi retrieval."""
    prefixed = f"{QUERY_INSTRUCTION}{query}"
    return embed_model.encode(
        prefixed,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

# Test nhanh
test_vec = embed_passages(["Điều 1. Phạm vi điều chỉnh"])
print(f"✅ embed_passages OK — shape: {test_vec.shape}")
test_q = embed_query("quy định về hòa giải ở cơ sở")
print(f"✅ embed_query OK   — shape: {test_q.shape}")

---
## 3. Kết nối ChromaDB Persistent

In [ ]:
# PersistentClient tự động lưu/load từ disk — không cần gọi .persist() thủ công
chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_DIR),
    settings=Settings(
        anonymized_telemetry=False,   # tắt telemetry
    )
)

print(f"✅ ChromaDB PersistentClient — path: {CHROMA_DIR}")
existing = chroma_client.list_collections()
print(f"   Collections hiện có: {[c.name for c in existing] if existing else '(trống)'}")

---
## 4. Utility Functions

In [ ]:
def sanitize_chroma_id(raw_id: str) -> str:
    """
    ChromaDB ID phải là string không có ký tự đặc biệt.
    Giữ nguyên chunk_id gốc trong metadata để truy xuất ngược.
    """
    # Thay / \ : * ? " < > | bằng _
    return re.sub(r'[/\\:*?"<>|]', '_', raw_id)


def get_or_create_collection(name: str, reset: bool = False) -> chromadb.Collection:
    """
    Lấy collection nếu đã tồn tại, hoặc tạo mới.
    reset=True: xóa và tạo lại (dùng khi re-index toàn bộ).
    """
    if reset:
        try:
            chroma_client.delete_collection(name)
            print(f"  🗑️  Đã xóa collection cũ: {name}")
        except Exception:
            pass

    collection = chroma_client.get_or_create_collection(
        name=name,
        # ChromaDB tự tính cosine nếu dùng normalized vectors
        metadata={"hnsw:space": "cosine"},
    )
    return collection


def index_chunks(
    df: pd.DataFrame,
    collection: chromadb.Collection,
    meta_cols: List[str],
    batch_size: int = 500,
    reset: bool = False,
) -> int:
    """
    Embed và nạp chunks vào ChromaDB.

    Args:
        df        : DataFrame chunks (phải có cột 'chunk_id', 'text')
        collection: ChromaDB collection đích
        meta_cols : Danh sách cột metadata cần lưu vào Chroma
        batch_size: Số chunk mỗi lần upsert
        reset     : True → bỏ qua chunk đã có, upsert tất cả

    Returns:
        Số chunk đã nạp
    """
    # Lọc chunk chưa có trong collection (incremental indexing)
    if not reset and collection.count() > 0:
        existing_ids = set(collection.get(include=[])['ids'])
        sanitized_ids = df['chunk_id'].apply(sanitize_chroma_id)
        mask = ~sanitized_ids.isin(existing_ids)
        df_new = df[mask].reset_index(drop=True)
        print(f"   Đã có: {len(existing_ids):,} | Mới cần nạp: {len(df_new):,}")
    else:
        df_new = df.reset_index(drop=True)

    if len(df_new) == 0:
        print("   ✅ Không có chunk mới — bỏ qua.")
        return 0

    total_indexed = 0
    n_batches = (len(df_new) + batch_size - 1) // batch_size

    for batch_i in tqdm(range(n_batches), desc=f"  Indexing {collection.name}"):
        start = batch_i * batch_size
        end   = min(start + batch_size, len(df_new))
        batch = df_new.iloc[start:end]

        texts = batch['text'].tolist()

        # Embed
        embeddings = embed_passages(texts, batch_size=EMBED_BATCH_SIZE)

        # IDs (sanitized cho Chroma, gốc giữ trong metadata)
        ids = batch['chunk_id'].apply(sanitize_chroma_id).tolist()

        # Metadata — ChromaDB chỉ chấp nhận str, int, float, bool
        # Không lưu 'text' vào metadata vì đã có documents
        metadatas = []
        for _, row in batch.iterrows():
            meta = {"chunk_id_raw": row['chunk_id']}  # giữ ID gốc (có /)
            for col in meta_cols:
                val = row.get(col, "")
                # Flatten list → JSON string (ChromaDB không nhận list)
                if isinstance(val, (list, np.ndarray)):
                    meta[col] = json.dumps(val, ensure_ascii=False)
                elif val is None or (isinstance(val, float) and np.isnan(val)):
                    meta[col] = ""
                else:
                    meta[col] = str(val)
            metadatas.append(meta)

        # Upsert vào Chroma
        collection.upsert(
            ids=ids,
            embeddings=embeddings.tolist(),
            documents=texts,
            metadatas=metadatas,
        )
        total_indexed += len(batch)

    return total_indexed


print("✅ Utility functions sẵn sàng.")

---
## 5. Index Legal Chunks

**Metadata lưu vào Chroma:**  
`chunk_id_raw`, `doc_id`, `source_doc_no`, `ministry`, `type_normalized`, `doc_name`, `chapter_id`, `chapter_name`, `article`, `split_type`, `chunk_index`, `total_chunks`, `word_count`

In [ ]:
print("=" * 60)
print("INDEXING: legal_chunks")
print("=" * 60)

legal_df = pd.read_parquet(CHUNK_PATHS["legal"], engine="pyarrow")
print(f"✅ Đọc xong: {len(legal_df):,} chunks | cột: {list(legal_df.columns)}")

LEGAL_META_COLS = [
    "doc_id", "source_doc_no", "ministry", "type_normalized",
    "doc_name", "chapter_id", "chapter_name", "article",
    "split_type", "chunk_index", "total_chunks", "word_count",
]

legal_col = get_or_create_collection(
    COLLECTION_NAMES["legal"],
    reset=False,   # False = incremental; True = re-index toàn bộ
)
print(f"   Collection '{legal_col.name}': {legal_col.count():,} chunks hiện có")

t0 = time.time()
n = index_chunks(legal_df, legal_col, meta_cols=LEGAL_META_COLS)
elapsed = time.time() - t0

print(f"\n✅ Legal indexing xong!")
print(f"   Đã nạp  : {n:,} chunks  ({elapsed:.0f}s)")
print(f"   Tổng DB  : {legal_col.count():,} chunks")

---
## 6. Index Forms Chunks

In [ ]:
print("=" * 60)
print("INDEXING: forms_chunks")
print("=" * 60)

forms_df = pd.read_parquet(CHUNK_PATHS["forms"], engine="pyarrow")
print(f"✅ Đọc xong: {len(forms_df):,} chunks | cột: {list(forms_df.columns)}")

FORMS_META_COLS = [
    "doc_id", "form_id", "form_type", "purpose",
    "required_fields",   # list → sẽ được json.dumps
    "split_type", "chunk_index", "total_chunks", "word_count",
]

forms_col = get_or_create_collection(COLLECTION_NAMES["forms"], reset=False)
print(f"   Collection '{forms_col.name}': {forms_col.count():,} chunks hiện có")

t0 = time.time()
n = index_chunks(forms_df, forms_col, meta_cols=FORMS_META_COLS)
elapsed = time.time() - t0

print(f"\n✅ Forms indexing xong!  ({n:,} chunks | {elapsed:.1f}s)")
print(f"   Tổng DB: {forms_col.count():,} chunks")

---
## 7. Index Examples Chunks

In [ ]:
print("=" * 60)
print("INDEXING: examples_chunks")
print("=" * 60)

examples_df = pd.read_parquet(CHUNK_PATHS["examples"], engine="pyarrow")
print(f"✅ Đọc xong: {len(examples_df):,} chunks | cột: {list(examples_df.columns)}")

EXAMPLES_META_COLS = [
    "doc_id", "example_id", "form_id", "form_type",
    "scenario", "fields_json",
    "split_type", "chunk_index", "total_chunks", "word_count",
]

examples_col = get_or_create_collection(COLLECTION_NAMES["examples"], reset=False)
print(f"   Collection '{examples_col.name}': {examples_col.count():,} chunks hiện có")

t0 = time.time()
n = index_chunks(examples_df, examples_col, meta_cols=EXAMPLES_META_COLS)
elapsed = time.time() - t0

print(f"\n✅ Examples indexing xong!  ({n:,} chunks | {elapsed:.1f}s)")
print(f"   Tổng DB: {examples_col.count():,} chunks")

---
## 8. Báo cáo tổng kết

In [ ]:
print("=" * 60)
print("        BÁO CÁO INDEXING")
print("=" * 60)

total = 0
for key, col_name in COLLECTION_NAMES.items():
    col = chroma_client.get_collection(col_name)
    cnt = col.count()
    total += cnt
    print(f"  {col_name:<25}: {cnt:>8,} chunks")

print(f"  {'─'*40}")
print(f"  {'TỔNG CỘNG':<25}: {total:>8,} chunks")

# Dung lượng ChromaDB trên disk
db_size_mb = sum(f.stat().st_size for f in CHROMA_DIR.rglob('*') if f.is_file()) / 1024 / 1024
print(f"\n  ChromaDB path : {CHROMA_DIR}")
print(f"  Dung lượng    : {db_size_mb:.1f} MB")

---
## 9. Hàm Retrieval & Kiểm tra

Phần này dùng để test — cũng là API sẽ gọi từ pipeline RAG.

In [ ]:
def retrieve_legal(
    query: str,
    top_k: int = 5,
    where: Optional[Dict] = None,
) -> List[Dict]:
    """
    Tìm kiếm điều khoản pháp luật liên quan.

    Args:
        query : Câu hỏi / mô tả nội dung cần tìm
        top_k : Số kết quả trả về
        where : Metadata filter, vd: {"type_normalized": "NGHỊ ĐỊNH"}
                hoặc lọc theo văn bản: {"source_doc_no": "01/2014/NQLT/CP-UBTƯMTTQVN"}

    Returns:
        List dict với keys: chunk_id_raw, text, article, doc_name,
                            source_doc_no, distance
    """
    query_vec = embed_query(query)

    results = legal_col.query(
        query_embeddings=[query_vec.tolist()],
        n_results=top_k,
        where=where,
        include=["documents", "metadatas", "distances"],
    )

    out = []
    for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0],
    ):
        out.append({
            "chunk_id_raw" : meta.get("chunk_id_raw", ""),
            "source_doc_no": meta.get("source_doc_no", ""),
            "doc_name"     : meta.get("doc_name", ""),
            "article"      : meta.get("article", ""),
            "type"         : meta.get("type_normalized", ""),
            "distance"     : round(dist, 4),
            "text"         : doc,
        })
    return out


def retrieve_forms(
    query: str,
    top_k: int = 3,
    where: Optional[Dict] = None,
) -> List[Dict]:
    """Tìm biểu mẫu phù hợp theo mô tả nhu cầu."""
    query_vec = embed_query(query)
    results = forms_col.query(
        query_embeddings=[query_vec.tolist()],
        n_results=top_k,
        where=where,
        include=["documents", "metadatas", "distances"],
    )
    out = []
    for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0],
    ):
        out.append({
            "chunk_id_raw": meta.get("chunk_id_raw", ""),
            "form_id"     : meta.get("form_id", ""),
            "form_type"   : meta.get("form_type", ""),
            "purpose"     : meta.get("purpose", ""),
            "distance"    : round(dist, 4),
            "text"        : doc,
        })
    return out


def retrieve_examples(
    query: str,
    top_k: int = 3,
    form_id: Optional[str] = None,
) -> List[Dict]:
    """Tìm ví dụ điền sẵn tương tự (few-shot retrieval)."""
    query_vec = embed_query(query)
    where = {"form_id": form_id} if form_id else None
    results = examples_col.query(
        query_embeddings=[query_vec.tolist()],
        n_results=top_k,
        where=where,
        include=["documents", "metadatas", "distances"],
    )
    out = []
    for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0],
    ):
        out.append({
            "chunk_id_raw": meta.get("chunk_id_raw", ""),
            "form_id"     : meta.get("form_id", ""),
            "scenario"    : meta.get("scenario", ""),
            "distance"    : round(dist, 4),
            "text"        : doc,
        })
    return out


print("✅ Các hàm retrieval đã định nghĩa.")

### 9.1. Test retrieval — Legal

In [ ]:
test_queries = [
    "quy định về hòa giải ở cơ sở",
    "điều kiện để được cấp giấy phép kinh doanh",
    "trách nhiệm của Bộ trưởng trong ban hành văn bản",
]

for q in test_queries:
    print(f"\n{'='*60}")
    print(f"🔍 Query: {q}")
    print(f"{'='*60}")
    results = retrieve_legal(q, top_k=3)
    for i, r in enumerate(results, 1):
        print(f"\n  [{i}] dist={r['distance']} | {r['source_doc_no']}")
        print(f"      {r['article']}")
        print(f"      {r['text'][:150]}...")

### 9.2. Test filter theo văn bản cụ thể

In [ ]:
# Truy xuất đúng điều khoản của 1 văn bản cụ thể
# Đây là use case: "trong văn bản X, điều nào nói về Y?"

DOC_NO = "01/2014/NQLT/CP-UBTƯMTTQVN"   # thay bằng số hiệu thực tế

results = retrieve_legal(
    query="phạm vi điều chỉnh",
    top_k=5,
    where={"source_doc_no": DOC_NO},   # filter metadata
)

print(f"🔍 Filter theo văn bản: {DOC_NO}")
print(f"   Kết quả: {len(results)} chunks")
for r in results:
    print(f"\n  dist={r['distance']} | {r['article']}")
    print(f"  {r['text'][:200]}...")

### 9.3. Test retrieval — Forms & Examples

In [ ]:
print("🔍 Forms retrieval:")
form_results = retrieve_forms("mẫu đơn xin việc làm", top_k=3)
for r in form_results:
    print(f"  [{r['form_id']}] dist={r['distance']} | {r['form_type']} — {r['purpose'][:80]}")

print("\n🔍 Examples retrieval:")
ex_results = retrieve_examples("đơn xin nghỉ phép", top_k=3)
for r in ex_results:
    print(f"  [{r['form_id']}] dist={r['distance']} | {r['scenario'][:80]}")
    print(f"  {r['text'][:150]}...")

---
## 10. Load lại ChromaDB ở session sau

Đây là cách dùng ở các notebook / script khác trong pipeline:

In [ ]:
# ── Template dùng ở pipeline RAG ─────────────────────────────────────────────
# Copy đoạn này vào notebook retrieval / inference

"""
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from pathlib import Path

CHROMA_DIR   = Path("dataset/chromadb")
EMBED_MODEL  = "intfloat/multilingual-e5-large-instruct"

# Load model & DB (đọc từ disk, không re-embed)
embed_model  = SentenceTransformer(EMBED_MODEL, device="cuda")  # hoặc "cpu"
chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_DIR),
    settings=Settings(anonymized_telemetry=False),
)

legal_col    = chroma_client.get_collection("legal_chunks")
forms_col    = chroma_client.get_collection("forms_chunks")
examples_col = chroma_client.get_collection("examples_chunks")

print(f"legal   : {legal_col.count():,} chunks")
print(f"forms   : {forms_col.count():,} chunks")
print(f"examples: {examples_col.count():,} chunks")
"""

print("📋 Template load ChromaDB đã sẵn sàng copy.")